# Filtering s_dose — standard (non-disease-specific) dosing test cases

Test cases exercising the standard dose filtering logic used by the DOSAGE validator app. Demonstrates filtering by generic name, age, and administration route.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load the standard dose dataset
s_dose = pd.read_csv("s_dose.csv")

In [ ]:

def filter_s_dose(generic=None, age=None, route=None, age_unit='y'):
    """
    Filter s_dose by generic, age, and route.
    Returns all matching rows with standard dosing recommendations.
    """
    df = s_dose.copy()

    # Apply generic filter
    if generic:
        df = df[df['generic'].str.lower() == generic.lower()]
    
    # Apply route filter
    if route:
        df = df[df['route'].str.upper() == route.upper()]
    
    # Handle age filtering across 3 unit types
    if age is not None:
        age_y = age if age_unit == 'y' else age / 12 if age_unit == 'm' else age / 365
        age_m = age if age_unit == 'm' else age * 12 if age_unit == 'y' else age / 30.44
        age_d = age if age_unit == 'd' else age * 365 if age_unit == 'y' else age * 30.44

        def age_match(row):
            for min_col, max_col, val in [
                ('min_age_y', 'max_age_y', age_y),
                ('min_age_m', 'max_age_m', age_m),
                ('min_age_d', 'max_age_d', age_d),
            ]:
                if pd.notna(row[min_col]) and pd.notna(row[max_col]):
                    if float(row[min_col]) <= val <= float(row[max_col]):
                        return True
            return False
        
        df = df[df.apply(age_match, axis=1)]
    
    return df.reset_index(drop=True)

# ▶ Example usage:
# Filter for Ciprofloxacin, age 25 years, PO route
result = filter_s_dose(generic="Ciprofloxacin", age=25, route="PO", age_unit='y')

print(result)


         generic  min_age_d  max_age_d  min_age_m  max_age_m  min_age_y  \
0  Ciprofloxacin        NaN        NaN        NaN        NaN       12.0   

   max_age_y  min_weight  max_weight  min_dw_mg  max_dw_mg  min_dw_iu  \
0      120.0         NaN         NaN        NaN        NaN        NaN   

   max_dw_iu  limit_mg  limit_iu  min_dd_mg  max_dd_mg  min_dd_iu  max_dd_iu  \
0        NaN       NaN       NaN      500.0     1500.0        NaN        NaN   

  route  
0    PO  
